# Logistic Regression
Checking the weights for Dry- and Wet-proofing.
What's the difference?

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
data_UK = pd.read_csv('../../../data/raw/SCALAR_Coastal_Study_new_respondents_Wave_Five_UK.csv')

In [3]:
print(data_UK.columns.tolist())

['ID', 'Q0_place_UK', 'Q1_home_NL_UK', 'Q4_home_size_UK', 'Q5_home_tenure', 'Q5b_home_sell', 'Q6_home_costs', 'Q7_move_in', 'Q8_move_out', 'Q12_neighborhood_trust', 'Q13_neighborhood_community', 'Q14_neighborhood_pleasure', 'Q15_neighborhood_favorite', 'Q16_neighborhood_identity', 'Q11_search_improve', 'Q11_search_social', 'Q11_search_family', 'Q11_search_area', 'Q11_search_job', 'Q11_search_location', 'Q11_search_hazard', 'Q11_search_other', 'Q11_search_dont_know', 'Q11a_hazard_type1', 'Q11a_hazard_type3', 'Q11a_hazard_type4', 'Q11a_hazard_type5', 'Q11a_hazard_type10', 'Q11a_hazard_type6', 'Q11a_hazard_type9', 'Q11a_hazard_not_say', 'R02_perc_prob', 'R02_perc_prob_other_text', 'Q18_flood_exp', 'Q18a_flood_where', 'Q18b_flood_year', 'Q18d_flood_cost', 'R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3', 'R05_worry', 'R01_resilience_5', 'R01_resilience_6', 'Q17_compens_noone', 'Q17_compens_ins', 'Q17_compens_owner', 'Q17_compens_family', 'Q17_compens_ngo', 'Q17_compens_o

For logistic regression on whether households will take measures, we need the following data:
- threat appraisal:
    * perceived probability (R02_perc_prob)
    * perceived damage / severity (R03_perc_damage)
    * worry (R05_worry)
- coping appraisal
    * perceived cost (R1c_perc_costs)
    * perceived response efficacy (R1b_resp_efficacy)
    * self-efficacy (R1a_self_efficacy)
    * each for the following measures
       * dry-proofing:
          * installing anti-backflow valves on pipes (SM5)
          * installing a pump or similar to drain water (SM6)
          * fixing water barriers (SM7)
       * wet-proofing:
          * strengthening house foundations (SM2)
          * reinforcing walls (SM3)
          * raising electricity meter (SM4)
- other:
    * flood experience
 
And for the y: 
- R2_implementation for relevant measures. Reminder: 1: already implemented, 2: plan to implement in near future (next 1-3 years), rest: implement later or never)


In [4]:
relevant_columns = ['R02_perc_prob', 'R05_worry', 'R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3',# threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7', 
                    'Q18_flood_exp',
                    'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']

In [5]:
data_households = data_UK[relevant_columns]
data_households.head()

,R02_perc_prob,R05_worry,R03_perc_damage_UK1,R03_perc_damage_UK2,R03_perc_damage_UK3,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7
0,6,1,,98,,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4
1,2,1,,1,,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4
2,1,1,,1,,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4
3,1,1,,1,,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4
4,2,1,,1,,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4


Need to combine some columns. 
- make one perceived damage column --> combining the UK1, UK2, UK3 perceived damages (no overlap, so just copy it all into a new column)
- combining dry- and wet-proofing --> averages

In [6]:
data_households = data_households.replace(r'^\s*$', np.nan, regex=True)
data_households['R03_perc_damage_UK1'] = pd.to_numeric(data_households['R03_perc_damage_UK1'])
data_households['R03_perc_damage_UK2'] = pd.to_numeric(data_households['R03_perc_damage_UK2'])
data_households['R03_perc_damage_UK3'] = pd.to_numeric(data_households['R03_perc_damage_UK3'])
data_households['R03_perc_damage'] = data_households.R03_perc_damage_UK1.fillna(0) + data_households.R03_perc_damage_UK2.fillna(0)+ data_households.R03_perc_damage_UK3.fillna(0)

In [7]:
data_households['self_efficacy_DP'] = data_households[['R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7']].mean(axis=1)
data_households['self_efficacy_WP'] = data_households[['R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4']].mean(axis=1)
data_households['resp_efficacy_DP'] = data_households[['R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7',]].mean(axis=1)
data_households['resp_efficacy_WP'] = data_households[['R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4']].mean(axis=1)
data_households['perc_cost_DP'] = data_households[['R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7']].mean(axis=1)
data_households['perc_cost_WP'] = data_households[['R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4']].mean(axis=1)

In [8]:
data_households.head()

,R02_perc_prob,R05_worry,R03_perc_damage_UK1,R03_perc_damage_UK2,R03_perc_damage_UK3,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP
0,6,1,NaN,98.0,NaN,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4,98.0,3.000000,2.333333,1.333333,1.000000,2.666667,4.000000
1,2,1,NaN,1.0,NaN,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4,1.0,3.333333,2.333333,2.333333,1.000000,2.333333,4.333333
2,1,1,NaN,1.0,NaN,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4,1.0,3.666667,3.666667,4.333333,4.666667,1.666667,3.666667
3,1,1,NaN,1.0,NaN,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4,1.0,1.000000,1.000000,5.000000,5.000000,5.000000,5.000000
4,2,1,NaN,1.0,NaN,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4,1.0,2.666667,2.000000,2.000000,2.000000,5.000000,5.000000


How to combine the implementation for dry- and wet-proofing. Maybe they plan to take some of the dry-proofing emasures but not all. I don't think it makes sense to take the average.
Idea: always take the lowest number available in the three columns that go into the variable (assumption: if they would take 1, they would also do the other)

In [9]:
data_households['implement_DP'] = data_households[['R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']].min(axis=1)
data_households['implement_WP'] = data_households[['R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4']].min(axis=1)


Cleaning up: removing rows with don't know answers and removing the columns that were used to combine for wet- and dry-proofing just above. 

In [10]:
# remove rows with don't know
data_households_98_removed = data_households[(data_households.R02_perc_prob != 95) &
                                            (data_households.R02_perc_prob != 98) & 
                                            (data_households.R02_perc_prob != 97) & 
                                            (data_households.R03_perc_damage != 98) &
                                            (data_households.R05_worry != 98)
                                            ]
data_households_98_removed.describe()

,R02_perc_prob,R05_worry,R03_perc_damage_UK1,R03_perc_damage_UK2,R03_perc_damage_UK3,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
count,563.000000,563.000000,110.000000,210.000000,243.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.00000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000,563.000000
mean,2.341030,1.527531,1.800000,1.776190,2.345679,1.573712,1.726465,2.099467,2.156306,2.060391,1.92540,2.561279,2.580817,2.880995,2.644760,2.781528,2.632327,4.582593,4.445826,3.612789,3.682060,3.932504,3.923623,0.174067,3.845471,3.850799,3.692718,3.786856,3.843694,3.861456,2.026643,2.047365,1.799882,2.686205,2.674364,3.846063,4.213736,3.721137,3.646536
std,1.890472,0.868921,1.217608,1.036374,1.309474,1.029538,1.133347,1.403117,1.364422,1.286355,1.25195,1.264514,1.236365,1.362321,1.178811,1.210958,1.270309,0.827582,0.840583,1.201710,1.103286,1.048160,1.106207,0.379505,0.565239,0.549129,0.824276,0.634734,0.527272,0.525495,1.225907,1.168202,1.041742,1.100259,1.163119,0.953644,0.779523,0.734530,0.881463
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,4.000000,4.000000,3.000000,3.000000,3.000000,3.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,1.000000,1.000000,1.000000,2.000000,1.666667,3.000000,3.666667,4.000000,4.000000
50%,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,5.000000,5.000000,4.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,1.666667,1.333333,2.666667,2.666667,4.000000,4.333333,4.000000,4.000000
75%,3.000000,2.000000,2.000000,2.000000,3.000000,2.000000,2.000000,3.000000,3.000000,3.000000,3.00000,3.000000,3.000000,4.000000,3.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,3.000000,2.333333,3.333333,3.333333,5.000000,5.000000,4.000000,4.000000
max,8.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000,4.000000


In [11]:
data_households_98_removed = data_households_98_removed.drop(columns = ['R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3',# threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7',
                                          'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7'
                                                                       ])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
1,2,1,0,1.0,3.333333,2.333333,2.333333,1.000000,2.333333,4.333333,4,4
2,1,1,0,1.0,3.666667,3.666667,4.333333,4.666667,1.666667,3.666667,4,4
3,1,1,0,1.0,1.000000,1.000000,5.000000,5.000000,5.000000,5.000000,4,4
4,2,1,1,1.0,2.666667,2.000000,2.000000,2.000000,5.000000,5.000000,4,4
5,1,1,0,3.0,1.000000,1.000000,3.000000,3.000000,4.333333,4.666667,4,4


In [12]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()
# Normalize 
data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 
                            'self_efficacy_DP', 'self_efficacy_WP', 
                            'resp_efficacy_DP', 'resp_efficacy_WP', 
                            'perc_cost_DP', 'perc_cost_WP']] = scaler.fit_transform(data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 
                                                                                                                'self_efficacy_DP', 'self_efficacy_WP', 
                                                                                                                'resp_efficacy_DP', 'resp_efficacy_WP', 
                                                                                                                'perc_cost_DP', 'perc_cost_WP']])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
1,0.142857,0.0,0.0,0.0,0.583333,0.333333,0.333333,0.000000,0.333333,0.833333,4,4
2,0.000000,0.0,0.0,0.0,0.666667,0.666667,0.833333,0.916667,0.166667,0.666667,4,4
3,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,4,4
4,0.142857,0.0,1.0,0.0,0.416667,0.250000,0.250000,0.250000,1.000000,1.000000,4,4
5,0.000000,0.0,0.0,0.5,0.000000,0.000000,0.500000,0.500000,0.833333,0.916667,4,4


In [13]:
# binary columns for implementation
data_households_98_removed['done_DP'] = (data_households_98_removed['implement_DP'] == 1).astype(int)
data_households_98_removed['done_WP'] = (data_households_98_removed['implement_WP'] == 1).astype(int)
data_households_98_removed['plan_soon_DP'] = (data_households_98_removed['implement_DP'] == 2).astype(int)
data_households_98_removed['plan_soon_WP'] = (data_households_98_removed['implement_WP'] == 2).astype(int)
data_households_98_removed.head(10)

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP,done_DP,done_WP,plan_soon_DP,plan_soon_WP
1,0.142857,0.00,0.0,0.00,0.583333,0.333333,0.333333,0.000000,0.333333,0.833333,4,4,0,0,0,0
2,0.000000,0.00,0.0,0.00,0.666667,0.666667,0.833333,0.916667,0.166667,0.666667,4,4,0,0,0,0
3,0.000000,0.00,0.0,0.00,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,4,4,0,0,0,0
4,0.142857,0.00,1.0,0.00,0.416667,0.250000,0.250000,0.250000,1.000000,1.000000,4,4,0,0,0,0
5,0.000000,0.00,0.0,0.50,0.000000,0.000000,0.500000,0.500000,0.833333,0.916667,4,4,0,0,0,0
6,0.000000,0.00,0.0,0.00,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,4,4,0,0,0,0
7,0.000000,0.00,0.0,0.00,0.833333,0.000000,0.333333,0.166667,0.500000,0.916667,4,4,0,0,0,0
8,0.000000,0.25,0.0,0.00,0.583333,0.333333,0.583333,0.666667,0.500000,0.750000,4,1,0,1,0,0
9,0.000000,0.25,0.0,0.25,0.750000,0.166667,0.500000,0.000000,0.500000,1.000000,4,4,0,0,0,0
10,1.000000,1.00,1.0,0.50,0.000000,0.000000,0.500000,0.750000,1.000000,1.000000,4,4,0,0,0,0


In [14]:
data_DP_WP_reg = data_households_98_removed

In [15]:
data_DP_WP_reg.count()

R02_perc_prob       563
R05_worry           563
Q18_flood_exp       563
R03_perc_damage     563
self_efficacy_DP    563
self_efficacy_WP    563
resp_efficacy_DP    563
resp_efficacy_WP    563
perc_cost_DP        563
perc_cost_WP        563
implement_DP        563
implement_WP        563
done_DP             563
done_WP             563
plan_soon_DP        563
plan_soon_WP        563
dtype: int64

### Do the logistic regression

We do the logistic regression 4 times here. 
1. logistic regression to done_WP, so to the ones that have already taken the measure WP
2. logistic regression to plan_soon_WP: those who plan to take the measure in the next 6 months
3. logistic regression to done_DP, so to the ones that have already taken the measure DP
4. logistic regression to plan_soon_DP: those who plan to take the measure in the next 6 months

In [16]:
# Wet-proofing
# logistic regression to 'done_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_done = data_DP_WP_reg['done_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.91

Logistic Regression Equation:
logit(p) = -2.9395 (-0.4668 * R02_perc_prob) + (0.8956 * R05_worry) + (0.6750 * Q18_flood_exp) + (-0.7026 * R03_perc_damage) + (0.9506 * self_efficacy_WP) + (0.8039 * resp_efficacy_WP) + (-0.4628 * perc_cost_WP) + (2.2227 * done_DP)

Feature Weights: {'R02_perc_prob': -0.46676154997693076, 'R05_worry': 0.8956434135182495, 'Q18_flood_exp': 0.6749618189937894, 'R03_perc_damage': -0.7025662033112202, 'self_efficacy_WP': 0.9506463160391471, 'resp_efficacy_WP': 0.8038992388032845, 'perc_cost_WP': -0.46279650709862546, 'done_DP': 2.2226916601218005}
Intercept: -2.9395108286060982


In [17]:
LR_values_done_WP = dict(zip(feature_names, weights))
LR_values_done_WP['Intercept'] = intercept
LR_values_done_WP['Perc_probability'] = LR_values_done_WP['R02_perc_prob']
LR_values_done_WP['worry'] = LR_values_done_WP['R05_worry']
LR_values_done_WP['flood_experience'] = LR_values_done_WP['Q18_flood_exp']
LR_values_done_WP['perc_damage'] = LR_values_done_WP['R03_perc_damage']
LR_values_done_WP['self_efficacy'] = LR_values_done_WP['self_efficacy_WP']
LR_values_done_WP['resp_efficacy'] = LR_values_done_WP['resp_efficacy_WP']
LR_values_done_WP['perc_cost'] = LR_values_done_WP['perc_cost_WP']
LR_values_done_WP['done_other'] = LR_values_done_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_done_WP.pop(k, None)
LR_values_done_WP

{'Intercept': -2.9395108286060982,
 'Perc_probability': -0.46676154997693076,
 'worry': 0.8956434135182495,
 'flood_experience': 0.6749618189937894,
 'perc_damage': -0.7025662033112202,
 'self_efficacy': 0.9506463160391471,
 'resp_efficacy': 0.8038992388032845,
 'perc_cost': -0.46279650709862546,
 'done_other': 2.2226916601218005}

In [18]:
# Wet-proofing
# logistic regression to 'plan_soon_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_plan = data_DP_WP_reg['plan_soon_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.96

Logistic Regression Equation:
logit(p) = -3.3194 (-0.2247 * R02_perc_prob) + (1.8061 * R05_worry) + (-0.4956 * Q18_flood_exp) + (0.2014 * R03_perc_damage) + (1.3857 * self_efficacy_WP) + (0.8034 * resp_efficacy_WP) + (-1.2926 * perc_cost_WP) + (0.1546 * done_DP)

Feature Weights: {'R02_perc_prob': -0.22466348937404146, 'R05_worry': 1.8061222962303003, 'Q18_flood_exp': -0.49555178135627476, 'R03_perc_damage': 0.20141626075920663, 'self_efficacy_WP': 1.3856966614697006, 'resp_efficacy_WP': 0.8033603652964337, 'perc_cost_WP': -1.2925792638238152, 'done_DP': 0.15464219737225465}
Intercept: -3.319428452244022


In [19]:
LR_values_plan_soon_WP = dict(zip(feature_names, weights))
LR_values_plan_soon_WP['Intercept'] = intercept
LR_values_plan_soon_WP['Perc_probability'] = LR_values_plan_soon_WP['R02_perc_prob']
LR_values_plan_soon_WP['worry'] = LR_values_plan_soon_WP['R05_worry']
LR_values_plan_soon_WP['flood_experience'] = LR_values_plan_soon_WP['Q18_flood_exp']
LR_values_plan_soon_WP['perc_damage'] = LR_values_plan_soon_WP['R03_perc_damage']
LR_values_plan_soon_WP['self_efficacy'] = LR_values_plan_soon_WP['self_efficacy_WP']
LR_values_plan_soon_WP['resp_efficacy'] = LR_values_plan_soon_WP['resp_efficacy_WP']
LR_values_plan_soon_WP['perc_cost'] = LR_values_plan_soon_WP['perc_cost_WP']
LR_values_plan_soon_WP['done_other'] = LR_values_plan_soon_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_plan_soon_WP.pop(k, None)
LR_values_plan_soon_WP

{'Intercept': -3.319428452244022,
 'Perc_probability': -0.22466348937404146,
 'worry': 1.8061222962303003,
 'flood_experience': -0.49555178135627476,
 'perc_damage': 0.20141626075920663,
 'self_efficacy': 1.3856966614697006,
 'resp_efficacy': 0.8033603652964337,
 'perc_cost': -1.2925792638238152,
 'done_other': 0.15464219737225465}

In [20]:
# Dry-proofing
# logistic regression to 'done_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_done = data_DP_WP_reg['done_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 1.00

Logistic Regression Equation:
logit(p) = -4.9247 (-0.0394 * R02_perc_prob) + (0.2788 * R05_worry) + (-0.0626 * Q18_flood_exp) + (0.7224 * R03_perc_damage) + (0.8404 * self_efficacy_DP) + (0.5310 * resp_efficacy_DP) + (-0.2531 * perc_cost_DP) + (4.9725 * done_WP)

Feature Weights: {'R02_perc_prob': -0.03944086450968101, 'R05_worry': 0.27880656316095564, 'Q18_flood_exp': -0.0625771971616201, 'R03_perc_damage': 0.7224305959096456, 'self_efficacy_DP': 0.8403892874100007, 'resp_efficacy_DP': 0.5309569068693076, 'perc_cost_DP': -0.2531053505553226, 'done_WP': 4.972466190598134}
Intercept: -4.92471371403856


In [21]:
LR_values_done_DP = dict(zip(feature_names, weights))
LR_values_done_DP['Intercept'] = intercept
LR_values_done_DP['Perc_probability'] = LR_values_done_DP['R02_perc_prob']
LR_values_done_DP['worry'] = LR_values_done_DP['R05_worry']
LR_values_done_DP['flood_experience'] = LR_values_done_DP['Q18_flood_exp']
LR_values_done_DP['perc_damage'] = LR_values_done_DP['R03_perc_damage']
LR_values_done_DP['self_efficacy'] = LR_values_done_DP['self_efficacy_DP']
LR_values_done_DP['resp_efficacy'] = LR_values_done_DP['resp_efficacy_DP']
LR_values_done_DP['perc_cost'] = LR_values_done_DP['perc_cost_DP']
LR_values_done_DP['done_other'] = LR_values_done_DP['done_WP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_done_DP.pop(k, None)
LR_values_done_DP

{'Intercept': -4.92471371403856,
 'Perc_probability': -0.03944086450968101,
 'worry': 0.27880656316095564,
 'flood_experience': -0.0625771971616201,
 'perc_damage': 0.7224305959096456,
 'self_efficacy': 0.8403892874100007,
 'resp_efficacy': 0.5309569068693076,
 'perc_cost': -0.2531053505553226,
 'done_other': 4.972466190598134}

In [22]:
# Dry-proofing
# logistic regression to 'plan_soon_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_plan = data_DP_WP_reg['plan_soon_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.96

Logistic Regression Equation:
logit(p) = -3.9242 (0.7888 * R02_perc_prob) + (1.6095 * R05_worry) + (0.0262 * Q18_flood_exp) + (0.2123 * R03_perc_damage) + (1.9306 * self_efficacy_DP) + (0.6803 * resp_efficacy_DP) + (-0.6494 * perc_cost_DP) + (-0.8885 * done_WP)

Feature Weights: {'R02_perc_prob': 0.7888102580370645, 'R05_worry': 1.6094508659203886, 'Q18_flood_exp': 0.026211084876323627, 'R03_perc_damage': 0.21226132304988202, 'self_efficacy_DP': 1.9305521735742683, 'resp_efficacy_DP': 0.6802738874870592, 'perc_cost_DP': -0.6493859171729032, 'done_WP': -0.8885186615253271}
Intercept: -3.924176311666148


In [23]:
LR_values_plan_soon_DP = dict(zip(feature_names, weights))
LR_values_plan_soon_DP['Intercept'] = intercept
LR_values_plan_soon_DP['Perc_probability'] = LR_values_plan_soon_DP['R02_perc_prob']
LR_values_plan_soon_DP['worry'] = LR_values_plan_soon_DP['R05_worry']
LR_values_plan_soon_DP['flood_experience'] = LR_values_plan_soon_DP['Q18_flood_exp']
LR_values_plan_soon_DP['perc_damage'] = LR_values_plan_soon_DP['R03_perc_damage']
LR_values_plan_soon_DP['self_efficacy'] = LR_values_plan_soon_DP['self_efficacy_DP']
LR_values_plan_soon_DP['resp_efficacy'] = LR_values_plan_soon_DP['resp_efficacy_DP']
LR_values_plan_soon_DP['perc_cost'] = LR_values_plan_soon_DP['perc_cost_DP']
LR_values_plan_soon_DP['done_other'] = LR_values_plan_soon_DP['done_WP']

remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_plan_soon_DP.pop(k, None)
LR_values_plan_soon_DP

{'Intercept': -3.924176311666148,
 'Perc_probability': 0.7888102580370645,
 'worry': 1.6094508659203886,
 'flood_experience': 0.026211084876323627,
 'perc_damage': 0.21226132304988202,
 'self_efficacy': 1.9305521735742683,
 'resp_efficacy': 0.6802738874870592,
 'perc_cost': -0.6493859171729032,
 'done_other': -0.8885186615253271}

In [24]:
# combining the dictionaries to a dataframe
LogisticReg_PMT = pd.DataFrame([LR_values_done_WP, LR_values_plan_soon_WP, LR_values_done_DP, LR_values_plan_soon_DP], index=['done_WP', 'plan_soon_WP', 'done_DP', 'plan_soon_DP'])

# Transpose the DataFrame to have keys as index and dictionary names as columns
LogisticReg_PMT = LogisticReg_PMT.T

# Define the desired order of the index
desired_order = ['Intercept', 'worry', 'perc_damage', 'Perc_probability', 'flood_experience', 
                 'self_efficacy', 'resp_efficacy', 
                 'perc_cost', 'done_other'
                ]

# Reindex the DataFrame
LogisticReg_PMT = LogisticReg_PMT.reindex(desired_order)

LogisticReg_PMT

,done_WP,plan_soon_WP,done_DP,plan_soon_DP
Intercept,-2.939511,-3.319428,-4.924714,-3.924176
worry,0.895643,1.806122,0.278807,1.609451
perc_damage,-0.702566,0.201416,0.722431,0.212261
Perc_probability,-0.466762,-0.224663,-0.039441,0.788810
flood_experience,0.674962,-0.495552,-0.062577,0.026211
self_efficacy,0.950646,1.385697,0.840389,1.930552
resp_efficacy,0.803899,0.803360,0.530957,0.680274
perc_cost,-0.462797,-1.292579,-0.253105,-0.649386
done_other,2.222692,0.154642,4.972466,-0.888519


In [25]:
LogisticReg_PMT_done = LogisticReg_PMT[['done_WP', 'done_DP']]
LogisticReg_PMT_done = LogisticReg_PMT_done.rename(columns={"done_WP": "wet-proofing", "done_DP": "dry-proofing"})
LogisticReg_PMT_plan_soon = LogisticReg_PMT[['plan_soon_WP', 'plan_soon_DP']]
LogisticReg_PMT_plan_soon = LogisticReg_PMT_plan_soon.rename(columns={"plan_soon_WP": "wet-proofing", "plan_soon_DP": "dry-proofing"})
LogisticReg_PMT_done

,wet-proofing,dry-proofing
Intercept,-2.939511,-4.924714
worry,0.895643,0.278807
perc_damage,-0.702566,0.722431
Perc_probability,-0.466762,-0.039441
flood_experience,0.674962,-0.062577
self_efficacy,0.950646,0.840389
resp_efficacy,0.803899,0.530957
perc_cost,-0.462797,-0.253105
done_other,2.222692,4.972466


In [26]:
LogisticReg_PMT_done.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_done_UK.csv')
LogisticReg_PMT_plan_soon.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_plan_soon_UK.csv')